# 문항 1. 전자상거래 회원·상품 데이터 모델링

회원(User)과 상품(Product)의 데이터 구조를 인터페이스와 타입 별칭으로 정의하고
더미 데이터가 타입 규칙에 맞게 컴파일되는지 확인합니다.

In [17]:
// 셀 타입 정의

type UserRole = 'admin' | 'customer' | 'guest';
type Category = 'hamster-doll' | 'cat-doll' | 'otter-doll';

interface BaseEntity {
  id: number;
}

interface User extends BaseEntity {
  name: string;
  email: string;
  role: UserRole;
  isActive: boolean;
}

interface Product extends BaseEntity {
  title: string;
  price: number;
  category: Category;
  stock: number;
  discountRate?: number;
}

function formatWon(amount: number): string {
  return amount.toLocaleString('ko-KR') + '원';
}

In [ ]:
// 더미 데이터

const users: User[] = [
  { id: 1, name: '김관리', email: 'admin@test.com', role: 'admin', isActive: true },
  { id: 2, name: '이고객', email: 'lee@test.com', role: 'customer', isActive: true },
  { id: 3, name: '박고객', email: 'park@test.com', role: 'customer', isActive: false },
  { id: 4, name: '최손님', email: 'guest@test.com', role: 'guest', isActive: true },
];

const products: Product[] = [
  { id: 101, title: '치이카와', price: 80000, category: 'hamster-doll', stock: 15, discountRate: 0.1 },
  { id: 102, title: '하치와레', price: 80000, category: 'cat-doll', stock: 40 },
  { id: 103, title: '쿠리만쥬', price: 100000, category: 'otter-doll', stock: 8, discountRate: 0.2 },
];

console.log('users:', users.length, '명');
console.log('products:', products.length, '개');

users: 4 명
products: 3 개


## 타입 오류 확인과 수정

아래 셀은 일부러 잘못된 데이터를 넣어 컴파일러가 오류를 잡는지 확인합니다.
`@ts-expect-error`는 "다음 줄에서 타입 오류가 나야 정상"이라는 표시입니다.

In [ ]:
// 의도적 타입 오류

// @ts-expect-error 'vip'은 UserRole에 없는 값이라 오류가 나야 정상
const badUser: User = { id: 99, name: '오류01', email: 'error@test.com', role: 'vip', isActive: true };

// @ts-expect-error email 필드가 빠져 있어 오류가 나야 정상
const badUser2: User = { id: 98, name: '오류02', role: 'guest', isActive: true };

// @ts-expect-error price에 문자열을 넣어서 오류가 나야 정상
const badProduct: Product = { id: 999, title: '오류 상품', price: '천원', category: 'otter-doll', stock: 1 };

console.log('의도한 타입 오류 3개가 모두 컴파일러에 의해 잡혔습니다.');

의도한 타입 오류 3개가 모두 컴파일러에 의해 잡혔습니다.


In [ ]:
// 오류 수정

const fixedUser: User = { id: 99, name: '수정됨', email: 'fixed@test.com', role: 'customer', isActive: true };
const fixedProduct: Product = { id: 999, title: '수정된 상품', price: 1000, category: 'otter-doll', stock: 1 };
console.log(fixedUser.role, fixedProduct.price);

customer 1000


## 심화: 타입 안전 유틸리티 함수
`any`를 쓰지 않고 모든 매개변수와 반환값의 타입을 직접 지정한 함수 두 개로 작성합니다. 

In [ ]:
// 권한 기반 필터

function filterUsersByRole(users: User[], targetRole: UserRole): User[] {
  return users.filter((user) => user.role === targetRole);
}

In [ ]:
// 총 결제 금액 (undefined 방어)

function calculateTotalPrice(products: Product[]): number {
  return products.reduce((total, product) => {
    const rate = product.discountRate ?? 0;
    return total + product.price * (1 - rate);
  }, 0);
}

In [20]:
// 실행과 결과 확인

const customers = filterUsersByRole(users, 'customer');
console.log('customer:', customers.map((u) => u.name).join(', '));
console.log('admin:', filterUsersByRole(users, 'admin').map((u) => u.name).join(', '));
console.log('guest:', filterUsersByRole(users, 'guest').map((u) => u.name).join(', '));

const total = calculateTotalPrice(products);
console.log('총 결제 금액:', formatWon(total));
console.log('빈 배열:', formatWon(calculateTotalPrice([])));
console.log('할인 없는 상품만:', formatWon(calculateTotalPrice([products[1]])));

customer: 이고객, 박고객
admin: 김관리
guest: 최손님
총 결제 금액: 232,000원
빈 배열: 0원
할인 없는 상품만: 80,000원


### 잘못된 호출이 막히는지 확인

아래 셀은 일부러 잘못 호출합니다. `@ts-expect-error`는 "다음 줄은 오류가 나야 정상"이라는 표시이므로
이 셀이 에러 없이 끝나면 컴파일러가 세 가지 잘못된 호출을 모두 막았다는 의미입니다.

In [15]:
// 의도적 오류

// @ts-expect-error 'vip'은 UserRole이 아니므로 오류가 나야 정상
filterUsersByRole(users, 'vip');

// @ts-expect-error User[]를 Product[] 자리에 넘기면 오류가 나야 정상
calculateTotalPrice(users);

// @ts-expect-error 반환값은 User[]이므로 number 변수에 담으면 오류가 나야 정상
const wrong: number = filterUsersByRole(users, 'admin');

console.log('any 없이도 잘못된 호출 3개를 컴파일러가 막았습니다.');

any 없이도 잘못된 호출 3개를 컴파일러가 막았습니다.


### any를 쓰지 않은 근거

**1. 모든 매개변수와 반환값에 타입을 직접 명시**

    - `filterUsersByRole(users: User[], targetRole: UserRole): User[]`
    - `calculateTotalPrice(products: Product[]): number`

**2. any였다면 통과했을 잘못된 호출 제어**

    - `vip`(없는 역할), `User[]`를 상품 배열 자리에 전달하고 반환값을 `number`에 대입하는 경우를 위 셀에서 확인했습니다.

**3. undefined 가능성을 타입으로 드러내고 방어**

    - `discountRate?: number`는 `number | undefined`이므로 `?? 0`으로 방어한 뒤 계산했습니다. 할인율이 없는 상품과 빈 배열에서도 정상 동작합니다.

**4. 한계 확인**

    - tslab 기본 설정은 타입을 생략한 암묵적 any를 오류로 잡지 않기 때문에 컴파일러에만 맡기지 않고 코드에서 any를 직접 쓰지 않고 모든 타입을 명시하는 방식을 반영했습니다.